# Deep Learning: perché la profondità conta

Il codice del capitolo [«Deep Learning: perché la profondità conta»](https://book.paithon.it/main/DeepLearning/overview.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q numpy torch torchvision

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## Deep Learning: perché la profondità conta

[Leggi la pagina](https://book.paithon.it/main/DeepLearning/overview.html)


### Profondo, non solo largo


In [ ]:
from torch import nn

# in ingresso: un gruppo di immagini a colori, alte e larghe 128 pixel
model = nn.Sequential(
    nn.Conv2d(3, 32, 3), nn.ReLU(),    # primi strati: bordi e linee
    nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3), nn.ReLU(),   # strati intermedi: texture e parti
    nn.MaxPool2d(2),
    nn.Conv2d(64, 128, 3), nn.ReLU(),  # parti più grandi, oggetti
    nn.AdaptiveAvgPool2d(1),           # media di ogni foglio di risultati
    nn.Flatten(),
    nn.Linear(128, 10),                # la classe finale (un logit per classe)
)

## Reti convoluzionali (CNN)

[Leggi la pagina](https://book.paithon.it/main/DeepLearning/reti-convoluzionali.html)


### L'architettura tipica


In [ ]:
from torch import nn

# input: un batch di immagini in scala di grigi, shape (N, 1, 28, 28)
model = nn.Sequential(
    # blocco 1: 32 filtri 3x3, mappe grandi come l'input
    nn.Conv2d(1, 32, 3, padding="same"), nn.ReLU(),
    nn.MaxPool2d(2),               # 28x28 -> 14x14
    # blocco 2: più filtri man mano che le mappe rimpiccioliscono
    nn.Conv2d(32, 64, 3, padding="same"), nn.ReLU(),
    nn.MaxPool2d(2),               # 14x14 -> 7x7
    nn.Flatten(),                  # srotola in un vettore di 64*7*7 = 3136
    nn.Linear(64 * 7 * 7, 10),     # 10 classi (logit)
)

## Far funzionare le reti profonde

[Leggi la pagina](https://book.paithon.it/main/DeepLearning/ottimizzazione-regolarizzazione.html)


### Bersagli meno netti: il label smoothing


In [ ]:
import math

import numpy as np

K, eps, passo = 5, 0.1, 0.5

netto = np.zeros(K)
netto[0] = 1.0                   # gatto sì, tutto il resto no
morbido = np.full(K, eps / K)
morbido[0] += 1 - eps            # nove e due decimi al gatto, due decimi agli altri

def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()

def scendi(q, tappe):
    """Discesa del gradiente sui soli punteggi grezzi: la correzione vale p - q."""
    z = np.zeros(K)
    for t in range(1, max(tappe) + 1):
        z -= passo * (softmax(z) - q)
        if t in tappe:
            p = softmax(z)
            yield t, z[0] - z[1], p[0]

tappe = (10**3, 10**4, 10**5)
for (t, dn, pn), (_, dm, pm) in zip(scendi(netto, tappe), scendi(morbido, tappe)):
    print(f"{t:>6} passi | netto: distacco {dn:5.2f}, al gatto il {pn:8.4%}"
          f" | morbido: distacco {dm:4.2f}, al gatto il {pm:.4%}")

print("distacco previsto per il morbido:",
      round(math.log((K * (1 - eps) + eps) / eps), 2))

### Scendere bene: gli optimizer moderni


In [ ]:
import torch

def dopo_40_passi(momentum, weight_decay=0.0, a_mano=False):
    p = torch.nn.Parameter(torch.tensor([1.0]))
    opt = torch.optim.SGD([p], lr=0.1, momentum=momentum, weight_decay=weight_decay)
    for _ in range(40):
        opt.zero_grad(); p.grad = torch.zeros_like(p)   # gradiente nullo
        opt.step()
        if a_mano:                # il decadimento vero, fuori dall'ottimizzatore
            with torch.no_grad(): p.mul_(1 - 0.1 * 0.05)    # 1 - eta*lambda
    return p.item()

print(f"weight_decay, senza momentum: {dopo_40_passi(0.0, weight_decay=0.05):.4f}")
print(f"a mano,       senza momentum: {dopo_40_passi(0.0, a_mano=True):.4f}")
print(f"a mano,       momentum 0,9:   {dopo_40_passi(0.9, a_mano=True):.4f}")
print(f"weight_decay, momentum 0,9:   {dopo_40_passi(0.9, weight_decay=0.05):.4f}")

In [ ]:
for Opt in (torch.optim.SGD, torch.optim.AdamW, torch.optim.Adam):
    p = torch.nn.Parameter(torch.tensor([1.0]))
    opt = Opt([p], lr=0.5, weight_decay=0.1)
    opt.zero_grad(); p.grad = torch.zeros_like(p); opt.step()
    print(f"{Opt.__name__:5s} dopo un passo: {p.item():.2f}")

### Regolare il passo nel tempo


*Frammento illustrativo: nel libro mostra la forma, qui non si esegue.*

```python

from torch import nn, optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)   # passo iniziale

# dimezza il learning rate quando la loss di validazione smette di scendere
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer,
                                                 factor=0.5, patience=3)

for epoca in range(50):        # un'«epoca» è una passata su tutti i dati
    addestra_una_epoca(model, train_loader, criterion, optimizer)
    loss_val = valuta(model, val_loader, criterion)   # loss di validazione
    scheduler.step(loss_val)                          # decide se ridurre il passo
```


## Le architetture che hanno fatto la storia

[Leggi la pagina](https://book.paithon.it/main/DeepLearning/architetture-storiche.html)


### Separare lo spazio dai canali: la convoluzione che sta in un telefono


In [ ]:
import torch
import torch.nn as nn

C_IN, C_OUT, K, H, W = 64, 128, 3, 56, 56
x = torch.randn(1, C_IN, H, W)

# convoluzione ordinaria: ogni filtro guarda tutti i canali in una volta sola
ordinaria = nn.Conv2d(C_IN, C_OUT, K, padding=1, bias=False)

# separabile: prima la parte spaziale, un filtro per canale (groups=C_IN),
# poi la parte fra i canali, una 1x1 che li rimescola
separabile = nn.Sequential(
    nn.Conv2d(C_IN, C_IN, K, padding=1, groups=C_IN, bias=False),   # depthwise
    nn.Conv2d(C_IN, C_OUT, 1, bias=False),                          # pointwise
)

def parametri(m):
    return sum(p.numel() for p in m.parameters())

print("stessa forma in uscita:", ordinaria(x).shape == separabile(x).shape,
      tuple(separabile(x).shape))
print(f"parametri, ordinaria : {parametri(ordinaria):>8,}")
print(f"parametri, separabile: {parametri(separabile):>8,}")
print(f"risparmio            : {parametri(ordinaria) / parametri(separabile):.2f}x")

teorico = (K * K * C_OUT) / (K * K + C_OUT)
print(f"previsto dalla formula: {teorico:.2f}x   (limite: {K * K}x)")

## Una rete, molti compiti: l'apprendimento multi-compito

[Leggi la pagina](https://book.paithon.it/main/DeepLearning/multi-compito.html)


### In pratica: il guadagno si misura, e può essere negativo


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

D, H = 12, 64
N_ETICHETTATI, N_AUSILIARI, N_TEST = 40, 800, 2000   # poche etichette dove servono

def dati(seme):
    g = torch.Generator().manual_seed(seme)
    n = N_AUSILIARI + N_TEST
    X = torch.randn(n, D, generator=g)
    w = torch.randn(D, generator=g)
    nascosto = torch.tanh(X @ w)                 # la quantità che conta davvero
    return X, {
        "principale": nascosto,                  # etichettata solo su 40 esempi
        "parente":    nascosto ** 2,             # dipende dalla STESSA quantità
        # bersaglio che non dipende da X: nessuna rete può impararlo
        "rumore":     torch.randn(n, generator=g),
    }

def addestra(X, y, ausiliario, seme, passi=800):
    torch.manual_seed(seme)
    tronco = nn.Sequential(nn.Linear(D, H), nn.Tanh(), nn.Linear(H, H), nn.Tanh())
    teste = nn.ModuleDict({k: nn.Linear(H, 1) for k in y})
    ott = torch.optim.Adam(list(tronco.parameters()) + list(teste.parameters()),
                           lr=3e-3)
    for _ in range(passi):
        # il compito principale vede 40 esempi, l'ausiliario ne vede 800
        perdita = F.mse_loss(teste["principale"](tronco(X[:N_ETICHETTATI])).squeeze(-1),
                             y["principale"][:N_ETICHETTATI])
        if ausiliario:
            perdita = perdita + F.mse_loss(
                teste[ausiliario](tronco(X[:N_AUSILIARI])).squeeze(-1),
                y[ausiliario][:N_AUSILIARI])
        ott.zero_grad(); perdita.backward(); ott.step()
    with torch.no_grad():
        pred = teste["principale"](tronco(X[N_AUSILIARI:])).squeeze(-1)
        return F.mse_loss(pred, y["principale"][N_AUSILIARI:]).item()

base = None
# cinque semi per tre configurazioni: qualche minuto di attesa
for ausiliario in (None, "parente", "rumore"):
    errori = [addestra(*dati(s)[:2], ausiliario, s) for s in range(5)]
    media = sum(errori) / len(errori)
    if base is None:
        base = media
    nome = ausiliario or "nessuno"
    print(f"ausiliario: {nome:<10} errore sul test {media:.4f}"
          f"   ({100 * (media - base) / base:+.0f}%)")

## Imparare a imparare in fretta

[Leggi la pagina](https://book.paithon.it/main/DeepLearning/meta-apprendimento.html)


### In pratica: dieci punti su un'onda mai vista


In [ ]:
import torch

torch.set_num_threads(1)   # su una macchina carica i thread si ostacolano

def pesi(gen, misure=((1, 40), (40, 40), (40, 1))):
    """La rete come lista esplicita di tensori: serve perche' il ciclo interno
    deve produrre una lista NUOVA di parametri, senza toccare quella vecchia."""
    p = []
    for entra, esce in misure:
        w = torch.randn(entra, esce, generator=gen) * (2.0 / entra) ** 0.5
        p += [w.requires_grad_(), torch.zeros(esce, requires_grad=True)]
    return p

def rete(x, p):
    h = torch.relu(x @ p[0] + p[1])
    h = torch.relu(h @ p[2] + p[3])
    return h @ p[4] + p[5]

def compito(gen):
    """Un membro della famiglia: ampiezza e fase sorteggiate."""
    A = torch.rand(1, generator=gen) * 4.9 + 0.1
    fase = torch.rand(1, generator=gen) * torch.pi
    return lambda x: A * torch.sin(x + fase)

def punti(f, n, gen):
    x = torch.rand(n, 1, generator=gen) * 10 - 5
    return x, f(x)

def adatta(p, x, y, passi, alfa, grafo):
    """Il ciclo interno. Con grafo=True la catena resta derivabile, ed e' cio'
    che permette al ciclo esterno di derivare ATTRAVERSO l'adattamento."""
    for _ in range(passi):
        perdita = ((rete(x, p) - y) ** 2).mean()
        g = torch.autograd.grad(perdita, p, create_graph=grafo)
        p = [w - alfa * gw for w, gw in zip(p, g)]
    return p

ITER, LOTTO = 1000, 8

# --- meta-addestramento: si valuta il DOPO, non l'adesso
gen = torch.Generator().manual_seed(1)
maml = pesi(gen)
opt = torch.optim.Adam(maml, lr=1e-3)
for _ in range(ITER):
    perdita = 0.0
    for _ in range(LOTTO):
        f = compito(gen)
        xs, ys = punti(f, 10, gen)      # insieme di supporto
        xq, yq = punti(f, 10, gen)      # insieme di interrogazione
        adattati = adatta(maml, xs, ys, 1, 0.01, grafo=True)
        perdita = perdita + ((rete(xq, adattati) - yq) ** 2).mean()
    opt.zero_grad(); (perdita / LOTTO).backward(); opt.step()

# --- il termine di paragone: la stessa rete allenata su TUTTE le sinusoidi
gen2 = torch.Generator().manual_seed(1)
insieme = pesi(gen2)
opt2 = torch.optim.Adam(insieme, lr=1e-3)
for _ in range(ITER):
    perdita = 0.0
    for _ in range(LOTTO):
        f = compito(gen2)
        x, y = punti(f, 20, gen2)
        perdita = perdita + ((rete(x, insieme) - y) ** 2).mean()
    opt2.zero_grad(); (perdita / LOTTO).backward(); opt2.step()

# --- la prova: 100 sinusoidi mai viste, stesso adattamento per tutti e tre
import statistics
prova = torch.linspace(-5, 5, 200).reshape(-1, 1)
righe = {}
for etichetta, p0 in (("a caso", pesi(torch.Generator().manual_seed(3))),
                      ("allenata su tutte", insieme),
                      ("MAML", maml)):
    g = torch.Generator().manual_seed(7)      # le stesse 100 sinusoidi per tutti
    prima, dopo = [], []
    for _ in range(100):
        f = compito(g)
        xs, ys = punti(f, 10, g)
        with torch.no_grad():
            prima.append(((rete(prova, p0) - f(prova)) ** 2).mean().item())
        p1 = adatta([w.detach().requires_grad_() for w in p0],
                    xs, ys, 5, 0.01, grafo=False)
        with torch.no_grad():
            dopo.append(((rete(prova, p1) - f(prova)) ** 2).mean().item())
    righe[etichetta] = (statistics.median(prima), statistics.median(dopo),
                        sum(1 for a, b in zip(prima, dopo) if b < a))

print("errore quadratico mediano su 100 sinusoidi mai viste")
print("(mediano e non medio: una singola divergenza rende la media inutile)")
print(f"   {'':20s} {'prima':>8s} {'dopo 5 passi':>13s}   migliora in")
for etichetta, (a, b, quante) in righe.items():
    print(f"   {etichetta:20s} {a:8.2f} {b:13.2f}   {quante:3d} casi su 100")

with torch.no_grad():
    u = rete(prova, insieme)
    print(f"\nla rete allenata su tutte oscilla fra {u.min():.2f} e {u.max():.2f}:")
    print("e' la media della famiglia, non una sua sinusoide")